# Ранжирование поисковых результатов
Финальный пайплайн: Qwen 4B + Jina v3.5 + CatBoost.
Локальная среда: **Windows 10/11 x64, Python 3.13, NVIDIA CUDA 13.0**, CPU с AVX2.
Данные: `data/`. Результат: `output/submission.csv`.


## 1. Настройки
Задаю пути, версии и seed.


In [ ]:
from pathlib import Path
from collections import Counter
from functools import lru_cache
import gc
import hashlib
import html
import importlib.metadata as metadata
import inspect
import json
import os
import platform
import re
import subprocess
import sys
import time
import unicodedata
os.environ.update(PYTHONUTF8='1', PYTHONIOENCODING='utf-8', TOKENIZERS_PARALLELISM='false', HF_HUB_DISABLE_XET='1', OMP_NUM_THREADS='2', MKL_NUM_THREADS='2', CUBLAS_WORKSPACE_CONFIG=':4096:8')
import joblib
import numpy as np
import pandas as pd
import torch
from catboost import CatBoostRanker, Pool
from sklearn.feature_extraction.text import TfidfVectorizer
DATA_DIR = Path('data').resolve()
OUTPUT_DIR = Path('output').resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SCORES_DIR = OUTPUT_DIR / 'neural_scores'
SCORES_DIR.mkdir(exist_ok=True)
GPU_IDS = list(range(torch.cuda.device_count()))
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
CORE_VERSIONS = {'numpy': '2.3.5', 'pandas': '2.2.3', 'scipy': '1.17.0', 'scikit-learn': '1.8.0', 'snowballstemmer': '2.2.0', 'catboost': '1.2.10', 'joblib': '1.5.3', 'threadpoolctl': '3.6.0', 'torch': '2.11.0+cu130', 'transformers': '4.57.3', 'tokenizers': '0.22.1', 'huggingface-hub': '0.36.2', 'accelerate': '1.15.0', 'safetensors': '0.8.0'}
assert platform.system() == 'Windows', 'Используйте Windows 10/11 x64.'
assert sys.version_info[:2] == (3, 13), 'Используйте Python 3.13 x64.'
for package, version in CORE_VERSIONS.items():
    assert metadata.version(package) == version, f'Установите requirements.txt: {package}'
print(platform.system(), platform.python_version())
print('GPU:', [torch.cuda.get_device_name(i) for i in GPU_IDS])


## 2. Данные
Читаю идентификаторы строками и проверяю исходные файлы.


In [ ]:
def digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()
INPUT_HASHES = {'train.csv': '2c1b26c5d3ee642f2c959b758cea49d4176ca4c0dfe92fbb7236733eff801042', 'test.csv': '4667c49296b72bf64767338c5345cb2f6c7cfbd0671364e5b728d9d5bcc2bf20', 'sample_submission.csv': 'b2cd50bc4fbaa41cacf04a74d01746c0e9ca391c4e5124822278734700cc3090'}
for name, expected in INPUT_HASHES.items():
    assert (DATA_DIR / name).is_file(), f'Положите {name} в {DATA_DIR}'
    assert digest(DATA_DIR / name) == expected, f'Изменён исходный файл: {name}'
train = pd.read_csv(DATA_DIR / 'train.csv', dtype=str, keep_default_na=False)
test = pd.read_csv(DATA_DIR / 'test.csv', dtype=str, keep_default_na=False)
sample = pd.read_csv(DATA_DIR / 'sample_submission.csv', dtype={'id': str})
train['relevance'] = train.relevance.astype(int)
assert (len(train), train.query_id.nunique()) == (31530, 3042)
assert (len(test), test.query_id.nunique()) == (13507, 1305)
assert train.id.is_unique and test.id.is_unique and sample.id.is_unique
assert set(train.id).isdisjoint(test.id)
assert set(train.query_id).isdisjoint(test.query_id)
assert set(sample.id) == set(test.id)
assert train.relevance.isin([0, 1]).all()
print('Train:', train.shape, 'Test:', test.shape)


## 3. Очистка
Исправляю кодировку, HTML и пропуски. Числа и отрицания сохраняю.


In [ ]:
_MOJI_RE = re.compile('(?:Р[°±²³µ¶·]|С[Ѓ‚„…†‡])|(?:[РС][\\u0400-\\u04ff]){3,}')
_INVISIBLE_RE = re.compile('[\\u200b-\\u200f\\u202a-\\u202e\\u2060\\ufeff]')
_HTML_RE = re.compile('</?(?:p|br|div|span|b|i|em|strong|a|sup|sub|li|ul|ol)(?:\\s[^<>]*?)?\\s*/?>', flags=re.I)
_TOKEN_RE = re.compile('[^\\W_]+', flags=re.UNICODE)
_EMPTY_TITLE = {'', 'null', 'n/a', 'не указано', '—'}
_STOP = set('а и но или в во на по из с со за от до к ко у о об это был была были быть есть как какой какая какие какое кто что где когда куда откуда почему зачем сколько чем чего чья чье чей чьи какого каком какую который которой которого которых ли'.split())

def _as_text(value):
    if value is None or value is pd.NA:
        return ''
    if isinstance(value, (float, np.floating)) and np.isnan(value):
        return ''
    return str(value)

def repair_encoding(value):
    value = _as_text(value)
    if not _MOJI_RE.search(value):
        return value
    try:
        repaired = value.encode('cp1251').decode('utf-8')
    except (UnicodeEncodeError, UnicodeDecodeError):
        return value
    if len(_MOJI_RE.findall(repaired)) < len(_MOJI_RE.findall(value)):
        return repaired
    return value

def clean_text(value, title=False):
    value = repair_encoding(value)
    for _ in range(2):
        value = html.unescape(value)
    value = _HTML_RE.sub(' ', value)
    value = _INVISIBLE_RE.sub('', value)
    value = unicodedata.normalize('NFKC', value)
    value = value.replace('́', '').replace('̀', '')
    value = ' '.join(value.split())
    if title:
        value = re.sub('\\s+[-—–]\\s*википедия\\s*$', '', value, flags=re.I)
        if value.casefold() in _EMPTY_TITLE:
            value = ''
    return value

def parse_numeric(series):
    values = series.map(_as_text).str.replace('\\s', '', regex=True)
    return pd.to_numeric(values.str.replace(',', '.', regex=False), errors='coerce')


## 4. Нейросетевые оценки
Модели получают только тексты. Готовые расчёты сохраняю для повторного запуска.


In [ ]:
QWEN_MODEL = 'Qwen/Qwen3-Reranker-4B'
QWEN_REVISION = '22e683669bc0f0bd69640a1354a6d0aebcfeede5'
JINA_MODEL = 'jinaai/jina-reranker-v3.5'
JINA_REVISION = 'e8a93f33f0b22108f8c2364f8484ce3422552fbc'
QWEN_COLUMNS = ['qwen_max', 'qwen_first', 'qwen_mean', 'qwen_min', 'qwen_std', 'qwen_chunks']
JINA_COLUMNS = ['jina_view0', 'jina_view1', 'jina_mean']
neural_input = pd.concat([train, test], ignore_index=True)[['id', 'query_id', 'query', 'title', 'text']]
for column in ['query', 'title', 'text']:
    neural_input[column] = neural_input[column].map(lambda value: clean_text(value, title=column == 'title'))

def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')
    temporary.replace(path)

def cache_spec(stage, source):
    return {'input_hashes': INPUT_HASHES, 'stage': stage, 'model': QWEN_MODEL if stage == 'qwen' else JINA_MODEL, 'revision': QWEN_REVISION if stage == 'qwen' else JINA_REVISION, 'source_sha256': hashlib.sha256(source.encode()).hexdigest(), 'versions': CORE_VERSIONS}

def cached_scores(stage, spec):
    path = SCORES_DIR / f'{stage}.csv'
    info = SCORES_DIR / f'{stage}.json'
    if not path.exists() and (not info.exists()):
        return None
    assert path.exists() and info.exists(), f'Неполный кэш: {stage}'
    saved = json.loads(info.read_text(encoding='utf-8'))
    assert saved['spec'] == spec, f'Настройки изменились: выберите новый OUTPUT_DIR ({stage}).'
    assert saved['sha256'] == digest(path), f'Повреждён кэш: {stage}'
    values = pd.read_csv(path, dtype={'id': str})
    expected = QWEN_COLUMNS if stage == 'qwen' else JINA_COLUMNS
    assert values.columns.tolist() == ['id', *expected]
    assert values.id.is_unique and set(values.id) == set(neural_input.id)
    assert np.isfinite(values[expected].to_numpy()).all()
    return values

def save_scores(stage, spec, values):
    path = SCORES_DIR / f'{stage}.csv'
    temporary = path.with_suffix('.tmp')
    values.to_csv(temporary, index=False, float_format='%.17g', lineterminator='\n')
    temporary.replace(path)
    atomic_json(SCORES_DIR / f'{stage}.json', {'spec': spec, 'sha256': digest(path)})
    return cached_scores(stage, spec)

def worker_source(constants, funcs, entrypoint):
    imports = 'from pathlib import Path\nimport gc, hashlib, html, importlib.metadata, json, os, re, sys, time, unicodedata\nimport numpy as np\nimport pandas as pd\n'
    definitions = '\n\n'.join((inspect.getsource(fn) for fn in funcs))
    values = '\n'.join((f'{name} = {value!r}' for name, value in constants.items()))
    return imports + values + '\n\n' + definitions + '\n\n' + entrypoint + '\n'

def run_workers(script, arguments, stage, dependency_dir=None):
    if not GPU_IDS:
        raise RuntimeError('Для первого расчёта Qwen и Jina нужна NVIDIA GPU с CUDA.')
    jobs = list(arguments)
    for start in range(0, len(jobs), len(GPU_IDS)):
        processes, logs = ([], [])
        try:
            for gpu_id, (rank, args) in zip(GPU_IDS, jobs[start:start + len(GPU_IDS)]):
                env = os.environ.copy()
                env['CUDA_VISIBLE_DEVICES'] = str(gpu_id)
                env['PYTHONHASHSEED'] = str(SEED)
                env['PYTHONNOUSERSITE'] = '1'
                if dependency_dir is not None:
                    env['PYTHONPATH'] = str(dependency_dir)
                log = (OUTPUT_DIR / f'{stage}_{rank}.log').open('w', buffering=1, encoding='utf-8')
                logs.append(log)
                command = [sys.executable, '-u', str(script), *map(str, args)]
                processes.append(subprocess.Popen(command, env=env, stdout=log, stderr=subprocess.STDOUT))
            while any((process.poll() is None for process in processes)):
                for process in processes:
                    if process.poll() not in (None, 0):
                        raise RuntimeError(f'Ошибка {stage}: см. журнал в {OUTPUT_DIR}')
                for log in logs:
                    lines = Path(log.name).read_text(errors='replace', encoding='utf-8').splitlines()
                    if lines:
                        print(Path(log.name).name, lines[-1], flush=True)
                time.sleep(10)
            if any((process.returncode != 0 for process in processes)):
                raise RuntimeError(f'Ошибка {stage}: см. журнал в {OUTPUT_DIR}')
        finally:
            for process in processes:
                if process.poll() is None:
                    process.terminate()
                    process.wait(timeout=30)
            for log in logs:
                log.close()


## 5. Qwen
Покрываю длинный текст окнами. Для каждой пары сохраняю оценки всех окон.


In [ ]:
SCORER_VERSION = 'qwen3-selected-head-windows-v2'
PREFIX = '<|im_start|>system\nJudge whether the Document meets the requirements based on the Query and the Instruct provided. Note that the answer can only be "yes" or "no".<|im_end|>\n<|im_start|>user\n'
SUFFIX = '<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'
INSTRUCTION = 'Given a web search query, retrieve relevant passages that answer the query'


In [ ]:
def make_pair_chunks(tokenizer, query, title, text, max_length=1024, overlap=128, max_title_tokens=128, instruction=INSTRUCTION):
    if max_length < 1 or max_title_tokens < 0 or overlap < 0:
        raise ValueError('Token limits must be non-negative; max_length must be positive.')
    if not all((isinstance(value, str) for value in [query, title, text, instruction])):
        raise TypeError('Pass preprocessed string query/title/text, with missing values handled.')
    encode = lambda s: tokenizer.encode(s, add_special_tokens=False)
    prefix_ids = encode(PREFIX)
    suffix_ids = encode(SUFFIX)
    header = encode(f'<Instruct>: {instruction}\n<Query>: {query}\n<Document>: ')
    title_ids = encode(title)[:max_title_tokens]
    if title_ids:
        header += encode('Title: ') + title_ids + encode('\nText: ')
    text_ids = encode(text)
    budget = max_length - len(prefix_ids) - len(header) - len(suffix_ids)
    if budget <= overlap:
        raise ValueError('Query/title leaves insufficient document window; raise max_length.')
    windows = []
    start = 0
    while True:
        stop = min(start + budget, len(text_ids))
        tokens = prefix_ids + header + text_ids[start:stop] + suffix_ids
        assert len(tokens) <= max_length and tokens[-len(suffix_ids):] == suffix_ids
        windows.append(tokens)
        if stop == len(text_ids):
            return windows
        start += budget - overlap

def yes_no_token_ids(tokenizer):
    ids = [tokenizer.encode(s, add_special_tokens=False) for s in ['no', 'yes']]
    if not all((len(x) == 1 for x in ids)) or ids[0] == ids[1]:
        raise ValueError('Expected distinct, single-token no/yes labels.')
    return [ids[0][0], ids[1][0]]

def selected_head_margin(model, inputs, label_ids):
    import torch
    import torch.nn.functional as F
    if not hasattr(model, 'model'):
        raise TypeError('Pass an unwrapped Qwen3ForCausalLM with its .model decoder.')
    mask = inputs['attention_mask']
    if mask.ndim != 2 or mask.shape != inputs['input_ids'].shape:
        raise ValueError('Expected matching 2D input_ids and attention_mask.')
    if not bool(mask[:, -1].all()):
        raise ValueError('Last token must be real: use left padding.')
    positions = (mask.long().cumsum(-1) - 1).clamp_min(0)
    out = model.model(input_ids=inputs['input_ids'], attention_mask=mask, position_ids=positions, use_cache=False, return_dict=True)
    last = out.last_hidden_state[:, -1, :]
    head = model.get_output_embeddings()
    if type(head) is not torch.nn.Linear or not head.weight.is_floating_point():
        raise TypeError('Expected unquantized torch.nn.Linear output head.')
    rows = torch.tensor(label_ids, dtype=torch.long, device=head.weight.device)
    weight = head.weight.index_select(0, rows)
    bias = None if head.bias is None else head.bias.index_select(0, rows)
    logits = F.linear(last.to(device=weight.device, dtype=weight.dtype), weight, bias).float()
    margin = logits[:, 1] - logits[:, 0]
    if not bool(torch.isfinite(margin).all()):
        raise FloatingPointError('Non-finite Qwen score.')
    return margin


In [ ]:
def score_chunks(model, tokenizer, token_lists, batch_size=4, max_padded_tokens=4096):
    import numpy as np
    import torch
    if tokenizer.padding_side != 'left':
        raise ValueError('Set tokenizer.padding_side = left before calling.')
    if batch_size < 1 or max_padded_tokens < 1:
        raise ValueError('Positive batch_size and max_padded_tokens are required.')
    if any((not tokens or len(tokens) > max_padded_tokens for tokens in token_lists)):
        raise ValueError('Each non-empty window must fit max_padded_tokens.')
    label_ids = yes_no_token_ids(tokenizer)
    order = sorted(range(len(token_lists)), key=lambda i: (len(token_lists[i]), i))
    scores = np.empty(len(token_lists), dtype=np.float32)
    device = model.get_input_embeddings().weight.device
    model.eval()
    start = 0
    with torch.inference_mode():
        while start < len(order):
            stop = start + 1
            while stop < len(order) and stop - start < batch_size:
                if len(token_lists[order[stop]]) * (stop + 1 - start) > max_padded_tokens:
                    break
                stop += 1
            idx = order[start:stop]
            features = [{'input_ids': token_lists[i], 'attention_mask': [1] * len(token_lists[i])} for i in idx]
            batch = tokenizer.pad(features, padding=True, return_tensors='pt')
            batch = {key: val.to(device) for key, val in batch.items()}
            scores[idx] = selected_head_margin(model, batch, label_ids).cpu().numpy()
            start = stop
    if not np.isfinite(scores).all():
        raise FloatingPointError('Non-finite Qwen score array.')
    return scores

def score_rows_resumable(model, tokenizer, frame, cache_dir, *, model_id, revision, max_length=1024, overlap=128, max_title_tokens=128, instruction=INSTRUCTION, batch_size=4, max_padded_tokens=4096, block_rows=128, progress_callback=None):
    import hashlib
    import json
    import os
    from pathlib import Path
    import numpy as np
    import pandas as pd
    import torch
    import transformers
    import tokenizers
    columns = ['id', 'query', 'title', 'text']
    if not all((c in frame for c in columns)):
        raise ValueError('Missing required id/query/title/text column.')
    if frame[columns].isna().any().any() or not frame['id'].is_unique:
        raise ValueError('Inputs must have unique ids and no null text fields.')
    if block_rows < 1:
        raise ValueError('block_rows must be positive.')
    if not revision or len(revision) != 40:
        raise ValueError('Pin the full 40-character model revision.')
    rows = list(frame[columns].itertuples(index=False, name=None))
    if any((not all((isinstance(v, str) for v in row)) for row in rows)):
        raise TypeError('Read ids as strings; preprocess all text fields to strings.')
    digest = hashlib.sha256()
    for row in rows:
        digest.update((json.dumps(row, ensure_ascii=False, separators=(',', ':')) + '\n').encode())
    device = model.get_input_embeddings().weight.device
    device_name = torch.cuda.get_device_name(device) if device.type == 'cuda' else 'cpu'
    config = dict(scorer_version=SCORER_VERSION, model_id=model_id, revision=revision, input_sha256=digest.hexdigest(), rows=len(rows), max_length=max_length, overlap=overlap, max_title_tokens=max_title_tokens, instruction=instruction, prefix=PREFIX, suffix=SUFFIX, batch_size=batch_size, max_padded_tokens=max_padded_tokens, block_rows=block_rows, dtype=str(model.dtype), attention=getattr(model.config, '_attn_implementation', None), torch=torch.__version__, transformers=transformers.__version__, tokenizers=tokenizers.__version__, numpy=np.__version__, device_name=device_name, cuda=torch.version.cuda)
    config['fingerprint'] = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()
    cache = Path(cache_dir)
    cache.mkdir(parents=True, exist_ok=True)
    manifest_path, scores_path = (cache / 'manifest.json', cache / 'scores.npz')
    if scores_path.exists() and (not manifest_path.exists()):
        raise ValueError('Score cache without manifest; use a new cache directory.')
    if manifest_path.exists():
        previous = json.loads(manifest_path.read_text(encoding='utf-8'))
        if previous != config:
            raise ValueError('Cache input/config mismatch; use a new directory, do not mix scores.')
    else:
        temp = manifest_path.with_suffix('.json.tmp')
        temp.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8')
        os.replace(temp, manifest_path)
    features = np.full((len(rows), 6), np.nan, dtype=np.float32)
    if scores_path.exists():
        with np.load(scores_path, allow_pickle=False) as data:
            if str(data['fingerprint'].item()) != config['fingerprint']:
                raise ValueError('Score cache fingerprint mismatch.')
            features = data['features'].copy()
        if features.shape != (len(rows), 6):
            raise ValueError('Invalid score cache shape.')
        complete = np.isfinite(features).all(axis=1)
        if (features[complete, -1] < 1).any():
            raise ValueError('Invalid complete row with no document windows.')
    for block_start in range(0, len(rows), block_rows):
        block_stop = min(block_start + block_rows, len(rows))
        status = np.isfinite(features[block_start:block_stop]).all(axis=1)
        if status.all():
            continue
        if status.any():
            raise ValueError('Partly committed fixed block; refuse non-deterministic batch resume.')
        token_lists, slices = ([], [])
        for _, query, title, text in rows[block_start:block_stop]:
            chunks = make_pair_chunks(tokenizer, query, title, text, max_length, overlap, max_title_tokens, instruction)
            slices.append((len(token_lists), len(token_lists) + len(chunks)))
            token_lists.extend(chunks)
        values = score_chunks(model, tokenizer, token_lists, batch_size, max_padded_tokens)
        for offset, (left, right) in enumerate(slices):
            v = values[left:right]
            features[block_start + offset] = [v.max(), v[0], v.mean(), v.min(), v.std(), len(v)]
        temp = scores_path.with_suffix('.npz.tmp')
        with temp.open('wb') as handle:
            np.savez(handle, features=features, fingerprint=np.array(config['fingerprint']))
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temp, scores_path)
        if progress_callback:
            progress_callback(block_stop, len(rows))
        else:
            print(f'Qwen: {block_stop}/{len(rows)} rows completed', flush=True)
    if not np.isfinite(features).all():
        raise FloatingPointError('Incomplete Qwen result, no submission should be generated.')
    result = pd.DataFrame(features, columns=['qwen_max', 'qwen_first', 'qwen_mean', 'qwen_min', 'qwen_std', 'qwen_chunks'])
    result.insert(0, 'id', frame['id'].to_numpy())
    return result


In [ ]:
def run_qwen(rank, model_path, root):
    import os, sys, json, time
    from pathlib import Path
    import numpy as np
    import pandas as pd
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    root = Path(root)
    torch.set_num_threads(2)
    torch.manual_seed(42)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    tokenizer = AutoTokenizer.from_pretrained(model_path, padding_side='left')
    model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.float16, attn_implementation='sdpa', device_map={'': 'cuda:0'}, low_cpu_mem_usage=True).eval()
    frame = pd.read_pickle(root / f'input_{rank}.pkl')
    print('MODEL_READY', rank, len(frame), torch.cuda.get_device_name(0), flush=True)
    first = frame.iloc[0]
    tokens = make_pair_chunks(tokenizer, first['query'], first['title'], first['text'])[0]
    batch = tokenizer.pad([{'input_ids': tokens, 'attention_mask': [1] * len(tokens)}], padding=True, return_tensors='pt')
    batch = {k: v.to('cuda:0') for k, v in batch.items()}
    positions = (batch['attention_mask'].cumsum(-1) - 1).clamp_min(0)
    labels = yes_no_token_ids(tokenizer)
    with torch.inference_mode():
        optimized = selected_head_margin(model, batch, labels)
        logits = model(**batch, position_ids=positions, use_cache=False, logits_to_keep=1).logits[:, -1].float()
        reference = logits[:, labels[1]] - logits[:, labels[0]]
    torch.testing.assert_close(optimized, reference, atol=0.03, rtol=0.001)
    print('GPU_PROJECTION_CHECK', float((optimized - reference).abs().max()), flush=True)
    del batch, logits, optimized, reference
    started = time.monotonic()

    def progress(done, total):
        seconds = time.monotonic() - started
        print(json.dumps(dict(rank=rank, done=done, total=total, seconds=round(seconds, 1), rows_sec=round(done / max(seconds, 1), 3))), flush=True)
    scores = score_rows_resumable(model, tokenizer, frame, root / f'cache_{rank}', model_id='Qwen/Qwen3-Reranker-4B', revision=MODEL_REVISIONS['Qwen/Qwen3-Reranker-4B'], max_length=1024, overlap=128, max_title_tokens=128, batch_size=4, max_padded_tokens=4096, block_rows=128, progress_callback=progress)
    scores.to_csv(root / f'scores_{rank}.csv', index=False, float_format='%.17g', lineterminator='\n')
    print('SCORING_COMPLETE', rank, len(scores), flush=True)


In [ ]:
qwen_source = worker_source({'SCORER_VERSION': SCORER_VERSION, 'PREFIX': PREFIX, 'SUFFIX': SUFFIX, 'INSTRUCTION': INSTRUCTION, 'MODEL_REVISIONS': {QWEN_MODEL: QWEN_REVISION}}, [make_pair_chunks, yes_no_token_ids, selected_head_margin, score_chunks, score_rows_resumable, run_qwen], "if __name__ == '__main__': run_qwen(int(sys.argv[1]), sys.argv[2], sys.argv[3])")
qwen_spec = cache_spec('qwen', qwen_source)
qwen_scores = cached_scores('qwen', qwen_spec)
if qwen_scores is None:
    if not GPU_IDS:
        raise RuntimeError('Для расчёта Qwen нужна NVIDIA GPU с CUDA.')
    qwen_dependencies = OUTPUT_DIR / 'qwen_dependencies'
    env = os.environ.copy()
    env['PYTHONPATH'] = str(qwen_dependencies)
    probe = subprocess.run([sys.executable, '-c', 'import transformers, tokenizers; print(transformers.__version__, tokenizers.__version__)'], env=env, text=True, capture_output=True, encoding='utf-8')
    if probe.returncode != 0 or probe.stdout.strip() != '4.51.3 0.21.4':
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '--upgrade', '--target', str(qwen_dependencies), 'transformers==4.51.3', 'tokenizers==0.21.4'], check=True)
    from huggingface_hub import snapshot_download
    model_path = snapshot_download(QWEN_MODEL, revision=QWEN_REVISION, allow_patterns=['*.json', '*.safetensors', '*.txt', '*.model'], max_workers=4)
    script = OUTPUT_DIR / 'qwen_worker.py'
    script.write_text(qwen_source, encoding='utf-8')
    for rank in range(2):
        neural_input.iloc[rank::2][['id', 'query', 'title', 'text']].reset_index(drop=True).to_pickle(OUTPUT_DIR / f'input_{rank}.pkl')
    run_workers(script, [(rank, [rank, model_path, OUTPUT_DIR]) for rank in range(2)], 'qwen', qwen_dependencies)
    values = pd.concat([pd.read_csv(OUTPUT_DIR / f'scores_{rank}.csv', dtype={'id': str}) for rank in range(2)], ignore_index=True)
    qwen_scores = save_scores('qwen', qwen_spec, values[['id', *QWEN_COLUMNS]])
print('Qwen:', qwen_scores.shape)


## 6. Jina
Оцениваю полный список кандидатов в двух порядках и усредняю оценки.


In [ ]:
def input_hash(frame):
    columns = ['id', 'query_id', 'query', 'title', 'text']
    text = frame[columns].to_json(orient='records', force_ascii=False)
    return hashlib.sha256(text.encode()).hexdigest()

def load_jina(device):
    import torch
    from transformers import AutoModel, AutoTokenizer
    dtype = torch.float16 if str(device).startswith('cuda') else torch.float32
    model = AutoModel.from_pretrained(JINA_MODEL, revision=JINA_REVISION, code_revision=JINA_REVISION, trust_remote_code=True, torch_dtype=dtype, attn_implementation='sdpa', low_cpu_mem_usage=True).to(device).eval()
    model._tokenizer = AutoTokenizer.from_pretrained(JINA_MODEL, revision=JINA_REVISION, padding_side='left')
    return model

def score_jina(model, frame, output, reverse=True):
    import torch
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    config = {'model': JINA_MODEL, 'revision': JINA_REVISION, 'input_hash': input_hash(frame), 'rows': len(frame), 'queries': int(frame.query_id.nunique()), 'reverse': reverse, 'document_order': 'string id ascending; second view descending', 'dtype': str(next(model.parameters()).dtype), 'torch': torch.__version__, 'transformers': importlib.metadata.version('transformers'), 'attention': 'sdpa', 'implementation_hash': digest(__file__), 'utils_hash': digest(__file__), 'device': str(next(model.parameters()).device), 'device_name': torch.cuda.get_device_name() if next(model.parameters()).is_cuda else 'CPU'}
    manifest = output / 'manifest.json'
    if manifest.exists() and json.loads(manifest.read_text(encoding='utf-8')) != config:
        raise ValueError('Jina cache configuration changed; use a new directory')
    atomic_json(manifest, config)
    pieces = []
    started = time.perf_counter()
    for number, (query_id, group) in enumerate(frame.groupby('query_id', sort=True), 1):
        group = group.sort_values('id').copy()
        name = hashlib.sha256(query_id.encode()).hexdigest()[:24] + '.csv'
        path = output / name
        if path.exists():
            scores = pd.read_csv(path, dtype={'id': str})
        else:
            query = group['query'].iloc[0]
            docs = [f'Title: {r.title}\nText: {r.text}' if r.title else r.text for r in group.itertuples()]
            tokenizer = model._tokenizer
            lengths = [len(tokenizer.encode(d, add_special_tokens=False)) for d in docs]
            query_length = len(tokenizer.encode(query, add_special_tokens=False))
            if max(lengths) >= 8192 or query_length >= 1024:
                raise ValueError('A document exceeds the Jina full-text contract')
            if sum(lengths) + 2 * query_length + 256 + 16 * len(docs) > 131072:
                raise ValueError('Full candidate group exceeds model context')
            views = [np.arange(len(group))]
            if reverse:
                views.append(views[0][::-1])
            scores = pd.DataFrame({'id': group.id.to_numpy()})
            for view, order in enumerate(views):
                with torch.inference_mode():
                    ranked = model.rerank(query, [docs[i] for i in order])
                if len(ranked) != len(docs) or {int(r['index']) for r in ranked} != set(range(len(docs))):
                    raise ValueError('Jina did not return every candidate exactly once')
                restored = np.full(len(docs), np.nan)
                for row in ranked:
                    restored[order[int(row['index'])]] = float(row['relevance_score'])
                scores[f'jina_view{view}'] = restored
            scores['jina_mean'] = scores.filter(like='jina_view').mean(axis=1)
            if not np.isfinite(scores.drop(columns='id').to_numpy()).all():
                raise ValueError('Nonfinite Jina scores')
            temp = path.with_suffix('.tmp')
            scores.to_csv(temp, index=False, float_format='%.17g', lineterminator='\n')
            temp.replace(path)
        expected = group.id.tolist()
        if scores.id.tolist() != expected or not np.isfinite(scores.drop(columns='id')).all().all():
            raise ValueError('Invalid per-query cache')
        pieces.append(scores)
        if number % 100 == 0 or number == frame.query_id.nunique():
            print(json.dumps({'stage': 'jina', 'queries': number, 'total': int(frame.query_id.nunique()), 'elapsed_seconds': time.perf_counter() - started}), flush=True)
    combined = pd.concat(pieces, ignore_index=True).set_index('id').loc[frame.id].reset_index()
    assert combined.id.is_unique and len(combined) == len(frame)
    combined.to_csv(output / 'scores.csv', index=False, float_format='%.17g', lineterminator='\n')
    return combined


In [ ]:
def chunked_sdpa(attention, query, key, value, attn_mask=None, dropout_p=0.0, is_causal=False, scale=None, enable_gqa=False):
    import torch
    from torch.nn.attention import SDPBackend, sdpa_kernel
    if dropout_p != 0.0:
        raise ValueError('Chunked attention is for inference with dropout=0.')
    length, key_length = (query.shape[-2], key.shape[-2])
    chunks = []
    for start in range(0, length, 128):
        stop = min(start + 128, length)
        mask = attn_mask
        if mask is not None and mask.ndim >= 2 and (mask.shape[-2] == length):
            mask = mask[..., start:stop, :]
        if is_causal:
            causal = torch.arange(key_length, device=query.device)[None, :] <= torch.arange(start, stop, device=query.device)[:, None]
            if mask is None:
                mask = causal
            elif mask.dtype == torch.bool:
                mask = mask & causal
            else:
                mask = mask.masked_fill(~causal, float('-inf'))
        with sdpa_kernel(SDPBackend.MATH):
            chunks.append(attention(query[..., start:stop, :], key, value, attn_mask=mask, dropout_p=0.0, is_causal=False, scale=scale, enable_gqa=enable_gqa))
    return torch.cat(chunks, dim=-2)

def efficient_retry(model, output):
    import torch
    from functools import partial
    import transformers.integrations.sdpa_attention as integration
    from torch.nn.attention import SDPBackend, sdpa_kernel
    if importlib.metadata.version('transformers') != '4.57.3':
        raise ValueError('Memory adapter requires Transformers 4.57.3')
    generator = torch.Generator(device='cuda:0').manual_seed(42)
    q = torch.randn(1, 16, 256, 128, device='cuda:0', dtype=torch.float16, generator=generator)
    k = torch.randn(1, 8, 256, 128, device='cuda:0', dtype=torch.float16, generator=generator)
    v = torch.randn(1, 8, 256, 128, device='cuda:0', dtype=torch.float16, generator=generator)
    base_attention = torch.nn.functional.scaled_dot_product_attention
    reference = base_attention(q, k, v, is_causal=True, enable_gqa=True)
    try:
        with sdpa_kernel(SDPBackend.EFFICIENT_ATTENTION):
            observed = base_attention(q, k.repeat_interleave(2, dim=1), v.repeat_interleave(2, dim=1), is_causal=True)
        retry_backend = 'efficient'
    except RuntimeError:
        observed = chunked_sdpa(base_attention, q, k, v, is_causal=True, enable_gqa=True)
        retry_backend = 'chunked_math'
    torch.testing.assert_close(reference, observed, rtol=0.01, atol=0.003)
    protocol = {'retry_backend': retry_backend, 'query_chunk_size': 128, 'candidate_filtering': False, 'text_truncation_added': False, 'source_sha256': digest(__file__), 'torch': torch.__version__, 'transformers': importlib.metadata.version('transformers')}
    path = output / 'memory_protocol_windows.json'
    if path.exists() and json.loads(path.read_text(encoding='utf-8')) != protocol:
        raise ValueError('Memory execution protocol changed; use a new output directory.')
    atomic_json(path, protocol)
    del q, k, v, reference, observed
    torch.cuda.empty_cache()
    original = model.rerank
    original_forwards = [(layer, layer.forward) for layer in model.model.layers]

    def clean_capture_wrappers():
        for layer, forward in original_forwards:
            layer.forward = forward

    def rerank(query, documents, **kwargs):
        try:
            return original(query, documents, **kwargs)
        except torch.cuda.OutOfMemoryError:
            pass
        finally:
            clean_capture_wrappers()
        gc.collect()
        torch.cuda.empty_cache()
        old_selector = integration.use_gqa_in_sdpa
        integration.use_gqa_in_sdpa = lambda attention_mask, key: False
        try:
            if retry_backend == 'efficient':
                with sdpa_kernel(SDPBackend.EFFICIENT_ATTENTION):
                    result = original(query, documents, **kwargs)
            else:
                torch.nn.functional.scaled_dot_product_attention = partial(chunked_sdpa, base_attention)
                result = original(query, documents, **kwargs)
        finally:
            torch.nn.functional.scaled_dot_product_attention = base_attention
            integration.use_gqa_in_sdpa = old_selector
            clean_capture_wrappers()
        print(json.dumps({'stage': 'attention_retry', 'backend': retry_backend, 'documents': len(documents), 'all_candidates_retained': True}), flush=True)
        return result
    model.rerank = rerank
    return model

def run_jina(input_path, output_path):
    import torch
    output_path = Path(output_path)
    output_path.mkdir(parents=True, exist_ok=True)
    torch.set_num_threads(2)
    torch.manual_seed(42)
    inputs = pd.read_pickle(input_path)
    model = efficient_retry(load_jina('cuda:0'), output_path)
    score_jina(model, inputs, output_path)


In [ ]:
jina_source = worker_source({'JINA_MODEL': JINA_MODEL, 'JINA_REVISION': JINA_REVISION}, [digest, atomic_json, input_hash, load_jina, score_jina, chunked_sdpa, efficient_retry, run_jina], "if __name__ == '__main__': run_jina(sys.argv[1], sys.argv[2])")
jina_spec = cache_spec('jina', jina_source)
jina_scores = cached_scores('jina', jina_spec)
if jina_scores is None:
    script = OUTPUT_DIR / 'jina_worker.py'
    script.write_text(jina_source, encoding='utf-8')
    query_ids = sorted(neural_input.query_id.unique())
    arguments = []
    for rank in range(2):
        part = neural_input.loc[neural_input.query_id.isin(query_ids[rank::2])].reset_index(drop=True)
        input_path = OUTPUT_DIR / f'jina_input_{rank}.pkl'
        output_path = OUTPUT_DIR / f'jina_worker{rank}'
        part.to_pickle(input_path)
        arguments.append((rank, [input_path, output_path]))
    run_workers(script, arguments, 'jina')
    values = pd.concat([pd.read_csv(OUTPUT_DIR / f'jina_worker{rank}/scores.csv', dtype={'id': str}) for rank in range(2)], ignore_index=True)
    jina_scores = save_scores('jina', jina_spec, values[['id', *JINA_COLUMNS]])
print('Jina:', jina_scores.shape)


## 7. Лексика и метаданные
Считаю совпадения слов, длины, пропуски и признаки внутри запроса.


In [ ]:
def _lexical_features(query, title, text):
    q = _TOKEN_RE.findall(query.casefold().replace('ё', 'е'))
    t = _TOKEN_RE.findall(title.casefold().replace('ё', 'е'))
    d = _TOKEN_RE.findall(text.casefold().replace('ё', 'е'))
    qs, ts, ds = (set(q), set(t), set(d))
    content = qs - _STOP or qs
    nums = {w for w in qs if w.isdigit()}
    qb, tb, db = (set(zip(q, q[1:])), set(zip(t, t[1:])), set(zip(d, d[1:])))
    counts = Counter(d)
    qn, cn = (max(len(qs), 1), max(len(content), 1))
    qt, tt, dt = (' '.join(q), ' '.join(t), ' '.join(d))
    return {'query_tokens': len(q), 'title_tokens': len(t), 'text_tokens': len(d), 'query_chars': len(query), 'title_chars': len(title), 'text_chars': len(text), 'coverage_text': len(qs & ds) / qn, 'coverage_title': len(qs & ts) / qn, 'content_coverage_text': len(content & ds) / cn, 'content_coverage_title': len(content & ts) / cn, 'content_coverage_both': len(content & (ds | ts)) / cn, 'jaccard_text': len(qs & ds) / max(len(qs | ds), 1), 'jaccard_title': len(qs & ts) / max(len(qs | ts), 1), 'term_frequency_text': sum((counts[x] for x in qs)) / max(len(d), 1), 'bigram_coverage_text': len(qb & db) / max(len(qb), 1), 'bigram_coverage_title': len(qb & tb) / max(len(qb), 1), 'number_coverage_text': len(nums & ds) / max(len(nums), 1), 'query_has_number': int(bool(nums)), 'head_coverage': len(content & set(d[:40])) / cn, 'tail_coverage': len(content & set(d[-40:])) / cn, 'title_in_query': int(bool(tt) and tt in qt), 'title_in_text': int(bool(tt) and tt in dt), 'query_in_text': int(bool(qt) and qt in dt), 'query_has_negation': int('не' in qs or 'нет' in qs), 'text_has_negation': int('не' in ds or 'нет' in ds)}

def base_features(frame):
    required = {'query_id', 'article_id', 'query', 'title', 'text', 'passage_index', 'article_passage_count', 'article_char_count', 'ingest_source'}
    if not required.issubset(frame.columns):
        raise ValueError(f'Нет столбцов: {sorted(required - set(frame.columns))}')
    if frame.empty:
        raise ValueError('Пустой набор признаков')
    work = frame.reset_index(drop=True)
    raw = {c: work[c].map(_as_text) for c in ['query', 'title', 'text']}
    clean = {c: raw[c].map(lambda v: clean_text(v, title=c == 'title')) for c in raw}
    matrix = [_lexical_features(q, t, d) for q, t, d in zip(clean['query'], clean['title'], clean['text'])]
    result = pd.DataFrame(matrix)
    source = work.ingest_source.map(_as_text)
    pi = parse_numeric(work.passage_index)
    pc = parse_numeric(work.article_passage_count)
    cc = parse_numeric(work.article_char_count)
    for name, value in [('passage_index', pi), ('article_passage_count', pc), ('article_char_count', cc)]:
        result[name + '_raw'] = value
        result[name + '_missing'] = value.isna().astype(np.int8)
        result[name + '_negative'] = value.lt(0).astype(np.int8)
        result[name + '_zero'] = value.eq(0).astype(np.int8)
        result[name + '_fractional'] = (value.notna() & value.mod(1).ne(0)).astype(np.int8)
        result['log_' + name] = np.log1p(value.where(value >= 0))
    corrected = pi - source.eq('S3').astype(np.int8)
    result['passage_index_s3_hypothesis'] = corrected
    valid_count = pc.where(pc > 0)
    result['position_ratio_raw'] = pi / valid_count
    result['position_ratio_s3_hypothesis'] = corrected / valid_count
    result['position_outside_raw'] = ((pi < 0) | (pi >= pc) & (pc > 0)).astype(np.int8)
    result['position_outside_s3_hypothesis'] = ((corrected < 0) | (corrected >= pc) & (pc > 0)).astype(np.int8)
    result['article_chars_per_passage'] = cc.where(cc > 0) / valid_count
    result['article_chars_per_text'] = cc.where(cc > 0) / result.text_chars.clip(lower=1)
    result['article_chars_over_million'] = cc.gt(1000000.0).astype(np.int8)
    result['article_chars_less_than_text'] = cc.lt(result.text_chars).astype(np.int8)
    result['title_missing'] = clean['title'].eq('').astype(np.int8)
    result['text_encoding_repaired'] = raw['text'].map(repair_encoding).ne(raw['text']).astype(np.int8)
    result['text_has_html'] = raw['text'].str.contains('<[^>]+>|&(?:amp|lt|gt|nbsp|quot);', regex=True).astype(np.int8)
    result['text_has_invisible'] = raw['text'].str.contains(_INVISIBLE_RE).astype(np.int8)
    for value in ['S0', 'S1', 'S2', 'S3']:
        result['source_' + value] = source.eq(value).astype(np.int8)
    result['source_other'] = ~source.isin(['S0', 'S1', 'S2', 'S3'])
    lowered_q = clean['query'].str.casefold()
    for word in ['кто', 'что', 'где', 'когда', 'сколько', 'почему', 'как', 'какой', 'какая']:
        result['query_starts_' + word] = lowered_q.str.startswith(word + ' ').astype(np.int8)
    qid = work.query_id.astype(str)
    result['candidate_count'] = qid.map(qid.value_counts())
    result['article_count_in_query'] = work.groupby('query_id').article_id.transform('nunique')
    result['same_article_candidates'] = work.groupby(['query_id', 'article_id']).query_id.transform('size')
    for col in ['coverage_text', 'content_coverage_text', 'coverage_title', 'bigram_coverage_text', 'text_chars']:
        grouped = result[col].groupby(qid)
        result['relative_rank_' + col] = grouped.rank(method='average', pct=True)
        result['relative_center_' + col] = result[col] - grouped.transform('mean')
    result = result.replace([np.inf, -np.inf], np.nan).astype(np.float32)
    result.index = frame.index
    return result


## 8. TF-IDF и BM25
Обучаю статистики только на train. Фиксирую режим NumPy при выборе словарей.


In [ ]:
STOP = set('а и но или в во на по из с со за от до к ко у о об это был была были быть есть как какой какая какие какое кто что где когда куда откуда почему зачем сколько чем чего чья чье чей чьи какого каком какую который которой которого которых ли'.split())
TOKEN_RE = re.compile('[a-zа-яё0-9]+', re.I)
MOJI_RE = re.compile('(?:Р[°±²³µ¶·]|С[Ѓ‚„…†‡])|(?:[РС][\\u0400-\\u04ff]){3,}')
INVISIBLE_RE = re.compile('[\\u200b-\\u200f\\u202a-\\u202e\\u2060\\ufeff]')

def clean_for_features(value, title=False):
    value = '' if value is None or value is pd.NA else str(value)
    if MOJI_RE.search(value):
        try:
            repaired = value.encode('cp1251').decode('utf8')
            if len(MOJI_RE.findall(repaired)) < len(MOJI_RE.findall(value)):
                value = repaired
        except (UnicodeEncodeError, UnicodeDecodeError):
            pass
    for _ in range(2):
        value = html.unescape(value)
    value = re.sub('</?(?:p|br|div|span|b|i|em|strong|a|sup|sub|li|ul|ol)(?:\\s[^<>]*?)?\\s*/?>', ' ', value, flags=re.I)
    value = INVISIBLE_RE.sub('', value)
    value = unicodedata.normalize('NFKC', value).replace('́', '').replace('̀', '')
    value = ' '.join(value.split())
    if title:
        value = re.sub('\\s+[-—–]\\s*википедия\\s*$', '', value, flags=re.I)
        if value.casefold() in {'', 'null', 'n/a', 'не указано', '—'}:
            value = ''
    return value

@lru_cache(maxsize=200000)
def stem_word(word):
    import snowballstemmer
    if not hasattr(stem_word, '_stemmer'):
        stem_word._stemmer = snowballstemmer.stemmer('russian')
    return stem_word._stemmer.stemWord(word)

def prepare_text(frame, clean_fn=clean_for_features):
    out = pd.DataFrame(index=frame.index)
    for name in ['query', 'title', 'text']:
        text = frame[name].map(lambda x: clean_fn(x, title=name == 'title'))
        text = text.str.casefold().str.replace('ё', 'е', regex=False)
        out[name] = text
        out[name + '_tokens'] = text.map(TOKEN_RE.findall)
        out[name + '_stems'] = out[name + '_tokens'].map(lambda words: [stem_word(w) for w in words])
    return out

def _numeric_output(records, index):
    out = pd.DataFrame(records, index=index)
    return out.replace([np.inf, -np.inf], np.nan).astype(np.float32)

def fit_text_state(training, prepared=None):
    prep = prepare_text(training) if prepared is None else prepared.loc[training.index]
    unique = training.loc[~training.document_id.duplicated()].index
    corpus = (prep.loc[unique, 'title_stems'] + prep.loc[unique, 'text_stems']).map(' '.join)
    word = TfidfVectorizer(lowercase=False, tokenizer=str.split, token_pattern=None, ngram_range=(1, 2), min_df=2, max_features=50000, sublinear_tf=True, dtype=np.float32)
    word.fit(corpus)
    char = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=3, max_features=60000, sublinear_tf=True, dtype=np.float32)
    char.fit(prep.loc[unique, 'title'] + ' ' + prep.loc[unique, 'text'])
    docs = (prep.loc[unique, 'title_stems'] + prep.loc[unique, 'text_stems']).tolist()
    df = Counter((w for doc in docs for w in set(doc)))
    n = len(docs)
    idf = {w: float(np.log1p((n - v + 0.5) / (v + 0.5))) for w, v in df.items()}
    avg = {name: max(float(prep.loc[unique, name + '_stems'].map(len).mean()), 1.0) for name in ['title', 'text']}
    return {'word': word, 'char': char, 'idf': idf, 'average_lengths': avg, 'fit_rows': len(training), 'fit_documents': n, 'fit_ids_sha256': hashlib.sha256('\n'.join(sorted(training.id.astype(str))).encode()).hexdigest()}

def fit_text_state_local(training, prepared):
    inputs = OUTPUT_DIR / 'text_fit_input.pkl'
    state_path = OUTPUT_DIR / 'text_state.pkl'
    worker_path = OUTPUT_DIR / 'fit_text_state.py'
    joblib.dump((training[['id', 'document_id']], prepared[['title', 'text', 'title_stems', 'text_stems']]), inputs)
    imports = 'from collections import Counter\nimport hashlib, sys\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom sklearn.feature_extraction.text import TfidfVectorizer\n\n'
    entrypoint = "\nfeatures = np._core._multiarray_umath.__cpu_features__\nassert features['AVX2'], 'Для этого окружения нужен CPU с AVX2.'\nassert not features['AVX512_SKX'], 'Режим NumPy не применён.'\ntraining, prepared = joblib.load(sys.argv[1])\nstate = fit_text_state(training, prepared)\njoblib.dump(state, sys.argv[2])\n"
    worker_path.write_text(imports + inspect.getsource(fit_text_state) + entrypoint, encoding='utf-8')
    environment = os.environ.copy()
    environment['NPY_DISABLE_CPU_FEATURES'] = 'AVX512F,AVX512CD,AVX512_KNL,AVX512_KNM,AVX512_SKX,AVX512_CLX,AVX512_CNL,AVX512_ICL,AVX512_SPR'
    environment['PYTHONHASHSEED'] = str(SEED)
    environment['PYTHONNOUSERSITE'] = '1'
    result = subprocess.run([sys.executable, str(worker_path), str(inputs), str(state_path)], env=environment, text=True, capture_output=True, encoding='utf-8')
    (OUTPUT_DIR / 'text_fit.log').write_text(result.stdout + result.stderr, encoding='utf-8')
    if result.returncode:
        raise RuntimeError(result.stdout + result.stderr)
    inputs.unlink()
    return joblib.load(state_path)


In [ ]:
def transform_text_features(state, frame, prepared=None):
    prep = prepare_text(frame) if prepared is None else prepared.loc[frame.index]
    out = pd.DataFrame(index=frame.index)
    for mode, vectorizer, suffix in [('word', state['word'], '_stems'), ('char', state['char'], '')]:
        fields = {n: prep[n + suffix].map(' '.join) if suffix else prep[n] for n in ['query', 'title', 'text']}
        q = vectorizer.transform(fields['query'])
        for name in ['title', 'text']:
            d = vectorizer.transform(fields[name])
            out[f'v2_{mode}_tfidf_{name}'] = np.asarray(q.multiply(d).sum(axis=1)).ravel()
    idf = state['idf']
    bm = []
    for q, t, d in prep[['query_stems', 'title_stems', 'text_stems']].itertuples(index=False, name=None):
        row = {}
        uq = set(q)
        terms = sorted(uq)
        total = sum((idf.get(w, 0.0) for w in terms))
        row['v2_train_query_oov_fraction'] = sum((w not in idf for w in uq)) / max(len(uq), 1)
        for name, words in [('title', t), ('text', d)]:
            counts = Counter(words)
            denom = 1.5 * (0.25 + 0.75 * len(words) / state['average_lengths'][name])
            row[f'v2_train_bm25_{name}'] = sum((idf.get(w, 0.0) * counts[w] * 2.5 / (counts[w] + denom) for w in terms if w in counts))
            row[f'v2_train_idf_coverage_{name}'] = sum((idf.get(w, 0.0) for w in sorted(uq & counts.keys()))) / max(total, 1e-12)
        bm.append(row)
    out = pd.concat([out, _numeric_output(bm, frame.index)], axis=1)
    group_scores = pd.Series(index=frame.index, dtype=float)
    for _, g in frame.groupby('query_id', sort=False):
        docs = prep.loc[g.index, 'text_stems']
        n = len(docs)
        df = Counter((w for d in docs for w in set(d)))
        average = max(docs.map(len).mean(), 1.0)
        q = sorted(set(prep.loc[g.index[0], 'query_stems']))
        local_idf = {w: np.log1p((n - df[w] + 0.5) / (df[w] + 0.5)) for w in q}
        values = []
        for doc in docs:
            count = Counter(doc)
            den = 1.5 * (0.25 + 0.75 * len(doc) / average)
            values.append(sum((local_idf[w] * count[w] * 2.5 / (count[w] + den) for w in q if w in count)))
        group_scores.loc[g.index] = values
    out['v2_candidate_bm25_text'] = group_scores
    for name in ['v2_word_tfidf_text', 'v2_char_tfidf_text', 'v2_train_bm25_text', 'v2_candidate_bm25_text']:
        groups = out[name].groupby(frame.query_id)
        out[name + '_relative_rank'] = groups.rank(method='average', pct=True)
    return out.replace([np.inf, -np.inf], np.nan).astype(np.float32)


## 9. Матрица признаков
Объединяю 115 признаков и выравниваю строки по `id`.


In [ ]:
FORBIDDEN_FEATURES = {'id', 'query_id', 'document_id', 'article_id', 'relevance', 'fold', 'component', 'split', 'n_pos'}
WINDOWS = ['qwen_first', 'qwen_mean', 'qwen_min', 'qwen_std', 'qwen_chunks']

def _features(frame, values):
    if not isinstance(values, pd.DataFrame):
        raise TypeError('Features must be a DataFrame with explicit row indices')
    if not values.index.is_unique or not frame.index.is_unique:
        raise ValueError('Row indices must be unique')
    if not frame.index.isin(values.index).all():
        raise ValueError('Features are missing rows')
    if not values.columns.is_unique or FORBIDDEN_FEATURES & set(values.columns):
        raise ValueError('Duplicate or forbidden feature names')
    if len(values.columns) == 0:
        raise ValueError('No features')
    x = values.loc[frame.index].copy()
    if not all((pd.api.types.is_numeric_dtype(x[c]) for c in x)):
        raise TypeError('Features must be numeric; encode source/category explicitly')
    x = x.astype(np.float32)
    if np.isinf(x.to_numpy()).any():
        raise ValueError('Infinite features; handle invalid metadata first')
    return x

def _raw_scores(frame, values):
    if isinstance(values, pd.Series):
        if not values.index.is_unique or not frame.index.isin(values.index).all():
            raise ValueError('Qwen scores have missing or duplicate row indices')
        result = values.loc[frame.index].to_numpy(dtype=np.float64)
    else:
        result = np.asarray(values, dtype=np.float64)
    if result.shape != (len(frame),) or not np.isfinite(result).all():
        raise ValueError('Every selected row needs one finite Qwen raw score')
    return result

def add_qwen_features(frame, features, qwen_scores):
    x = _features(frame, features)
    score = pd.Series(_raw_scores(frame, qwen_scores), index=frame.index)
    groups = score.groupby(frame['query_id'])
    additions = {'qwen_raw': score, 'qwen_rank_fraction': groups.rank(method='average', pct=True), 'qwen_center': score - groups.transform('mean'), 'qwen_gap_to_max': score - groups.transform('max'), 'qwen_query_std': groups.transform('std').fillna(0)}
    if set(additions) & set(x):
        raise ValueError('Qwen feature names already present')
    for name, value in additions.items():
        x[name] = value.astype(np.float32)
    return x

def build_features(frame, qwen, jina, state, prepared):
    features = base_features(frame.drop(columns='relevance', errors='ignore'))
    assert len(features.columns) == 83
    for name in WINDOWS:
        features[name] = qwen[name].astype(np.float32)
    features = add_qwen_features(frame, features, qwen.qwen_max)
    assert len(features.columns) == 93
    extra = add_qwen_features(frame, pd.DataFrame({'temporary': 0.0}, index=frame.index), jina.jina_mean)
    extra = extra.drop(columns='temporary').add_prefix('jina_')
    extra['jina_forward'] = jina.jina_view0.astype(np.float32)
    extra['jina_reverse'] = jina.jina_view1.astype(np.float32)
    extra['jina_order_gap'] = (jina.jina_view0 - jina.jina_view1).abs().astype(np.float32)
    features = pd.concat([features, transform_text_features(state, frame, prepared), extra], axis=1)
    assert features.columns.is_unique and len(features.columns) == 115
    return features


In [ ]:
def align(frame, values, columns):
    result = values.set_index('id').loc[frame.id, columns].copy()
    result.index = frame.index
    assert np.isfinite(result.to_numpy()).all()
    return result
qtrain = align(train, qwen_scores, QWEN_COLUMNS)
qtest = align(test, qwen_scores, QWEN_COLUMNS)
jtrain = align(train, jina_scores, JINA_COLUMNS)
jtest = align(test, jina_scores, JINA_COLUMNS)
prepared_train = prepare_text(train)
prepared_test = prepare_text(test)
text_state = fit_text_state_local(train.drop(columns='relevance'), prepared_train)
xtrain = build_features(train, qtrain, jtrain, text_state, prepared_train)
xtest = build_features(test, qtest, jtest, text_state, prepared_test)
assert xtrain.columns.tolist() == xtest.columns.tolist()
assert xtrain.shape == (31530, 115) and xtest.shape == (13507, 115)
print(xtrain.shape, xtest.shape)


## 10. Обучение
На всём train обучаю CatBoost поправке к оценке Jina. Вес каждого запроса равен 1.


In [ ]:
def _query_pairs(ordered_frame):
    y = pd.to_numeric(ordered_frame['relevance'], errors='raise').to_numpy()
    if not np.isin(y, [0, 1]).all():
        raise ValueError('Expected binary competition labels')
    pairs, weights = ([], [])
    group_positions = ordered_frame.groupby('query_id', sort=False).indices
    for positions in group_positions.values():
        positive = positions[y[positions] == 1]
        negative = positions[y[positions] == 0]
        count = len(positive) * len(negative)
        if count:
            pairs.extend(((int(p), int(n)) for p in positive for n in negative))
            weights.extend([1.0 / count] * count)
    if not pairs:
        raise ValueError('No within-query positive-negative pairs to train on')
    return (np.asarray(pairs, dtype=np.int64), np.asarray(weights, dtype=np.float64))
base_train = 10.0 * jtrain.jina_mean
base_test = 10.0 * jtest.jina_mean
ordered = train.sort_values(['query_id', 'id'])
pairs, pair_weights = _query_pairs(ordered)
pool = Pool(xtrain.loc[ordered.index], label=ordered.relevance.to_numpy(), group_id=ordered.query_id.to_numpy(), baseline=base_train.loc[ordered.index].to_numpy().astype(np.float32), pairs=pairs, pairs_weight=pair_weights)
model = CatBoostRanker(loss_function='PairLogit', iterations=600, depth=6, learning_rate=0.03, l2_leaf_reg=5.0, random_seed=SEED, bootstrap_type='No', random_strength=0.0, task_type='CPU', thread_count=3, allow_writing_files=False, verbose=False)
model.fit(pool, use_best_model=False)
print('Деревьев:', model.tree_count_, 'Обучающих пар:', len(pairs))


## 11. Предсказания
Прибавляю обученную поправку к Jina и сохраняю все строки test.


In [ ]:
correction = np.asarray(model.predict(xtest), dtype=np.float64)
prediction = base_test.to_numpy(dtype=np.float64) + correction
assert np.isfinite(prediction).all()
centered = pd.Series(correction) - pd.Series(correction).groupby(test.query_id).transform('mean')
assert centered.abs().max() > 1e-10
submission = pd.DataFrame({'id': test.id, 'prediction': prediction})
submission = submission.set_index('id').loc[sample.id].reset_index()
assert submission.columns.tolist() == ['id', 'prediction']
assert submission.id.tolist() == sample.id.tolist()
assert len(submission) == 13507 and submission.id.is_unique
submission.to_csv(OUTPUT_DIR / 'submission.csv', index=False, float_format='%.17g', lineterminator='\n')
model.save_model(str(OUTPUT_DIR / 'model.cbm'))
joblib.dump(text_state, OUTPUT_DIR / 'text_state.pkl')
restored = CatBoostRanker().load_model(str(OUTPUT_DIR / 'model.cbm'))
np.testing.assert_allclose(restored.predict(xtest), correction, rtol=0, atol=1e-12)
print('Готово:', OUTPUT_DIR / 'submission.csv')
print('SHA256:', digest(OUTPUT_DIR / 'submission.csv'))
submission.head()
